# Pipeline Test
Test the full defense pipeline with different configurations (full_defense vs lightweight).

In [1]:
import os
import sys
import cv2

fas_root = os.path.abspath("../third_party/Silent-Face-Anti-Spoofing")
sys.path.insert(0, fas_root)
os.chdir(fas_root)

from face_defense.core.config import load_config
from face_defense.core.pipeline import build_pipeline

## 1. Lightweight Pipeline (Silent-FAS only)

In [2]:
config = load_config("../../configs/pipeline/lightweight.yaml")
pipeline = build_pipeline(config)

img_real = cv2.imread(os.path.abspath("../../notebooks/test_real.jpg"))
img_spoof = cv2.imread(os.path.abspath("../../notebooks/test_spoof.jpg"))
face_real = cv2.resize(img_real, (80, 80))
face_spoof = cv2.resize(img_spoof, (80, 80))

result = pipeline.run(face_real)
print(f"[Real] Verdict: {'REAL' if result.is_real else 'SPOOF'}, Confidence: {result.confidence:.4f}, Latency: {result.latency_ms:.1f}ms")

result = pipeline.run(face_spoof)
print(f"[Spoof] Verdict: {'REAL' if result.is_real else 'SPOOF'}, Confidence: {result.confidence:.4f}, Latency: {result.latency_ms:.1f}ms, Early exit: {result.exited_early}")

c:\Users\gmission\.conda\envs\face-defense\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\gmission\Desktop\face-defense\face_defense\models\anti_spoof\..\..\..\third_party\Silent-Face-Anti-Spoofing\src\anti_spoof_predict.py:91: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  result = F.softmax(result).cpu().numpy()


[Real] Verdict: REAL, Confidence: 0.9232, Latency: 185.6ms
[Spoof] Verdict: SPOOF, Confidence: 0.9919, Latency: 150.3ms, Early exit: True


## 2. Full Defense Pipeline (Silent-FAS + deepface)

In [3]:
config_full = load_config("../../configs/pipeline/full_defense.yaml")
pipeline_full = build_pipeline(config_full)

result = pipeline_full.run(face_real)
print(f"[Real] Verdict: {'REAL' if result.is_real else 'SPOOF'}, Confidence: {result.confidence:.4f}, Latency: {result.latency_ms:.1f}ms")

result = pipeline_full.run(face_spoof)
print(f"[Spoof] Verdict: {'REAL' if result.is_real else 'SPOOF'}, Confidence: {result.confidence:.4f}, Latency: {result.latency_ms:.1f}ms, Early exit: {result.exited_early}")

Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to C:\Users\gmission/.cache\torch\hub\checkpoints\xception-43020ad28.pth


[Real] Verdict: REAL, Confidence: 0.8258, Latency: 8006.2ms
[Spoof] Verdict: SPOOF, Confidence: 0.5083, Latency: 226.8ms, Early exit: False


## 3. Results Summary

| Pipeline | Real Image | Spoof Image | Latency |
|----------|-----------|-------------|---------|
| Lightweight | REAL (92.3%) | SPOOF (99.2%, early exit) | ~150ms |
| Full Defense | REAL (82.6%) | SPOOF (50.8%) | ~8000ms |

Lightweight pipeline is faster and more confident on spoof detection via early exit.
Full defense runs all 3 stages (anti-spoof + liveness + deepfake), resulting in lower confidence due to untuned model weights.